In [ ]:
# ============================================================
# CELL 1 - Environment Setup
# Install all required dependencies for MMSegmentation + SegFormer
# ============================================================

# Install PyTorch with CUDA 12.1 support
!pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121 -q

# Install OpenMMLab stack
!pip install mmengine -q
!pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html -q
!pip install mmsegmentation -q

# Fix mmcv version check (mmseg 1.2.2 hardcodes max version as 2.2.0 exclusive)
!sed -i 's/MMCV_MAX = .2.2.0./MMCV_MAX = "2.3.0"/' \
    /usr/local/lib/python3.12/dist-packages/mmseg/__init__.py

# Install text encoding dependencies
!pip install ftfy -q

# Install additional augmentation library
!pip install albumentations -q

print("Installation complete, version check fixed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 70.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 83.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 113.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 755.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# CELL 2 - Environment Verification
# Verify all packages are correctly installed and GPU is available
# ============================================================

import torch, mmcv, mmengine, mmseg

print('torch:', torch.__version__)
print('mmcv:', mmcv.__version__)
print('mmseg:', mmseg.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

torch: 2.4.1+cu121
mmcv: 2.2.0
mmseg: 1.2.2
CUDA available: True
GPU: Tesla T4


In [1]:
# ============================================================
# CELL 3 - Mount Google Drive and Extract Dataset
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Extract dataset from Drive to Colab local storage (faster I/O during training)
!unzip -q /content/drive/MyDrive/Dataset/processed.zip -d /content/

# Verify dataset structure and image counts
print("=== Dataset Verification ===")
!echo "Train images:" && ls /content/processed/train/images/ | wc -l
!echo "Train masks: " && ls /content/processed/train/masks/  | wc -l
!echo "Val images:  " && ls /content/processed/val/images/   | wc -l
!echo "Val masks:   " && ls /content/processed/val/masks/    | wc -l
!echo "Test images: " && ls /content/processed/test/images/  | wc -l
!echo "Test masks:  " && ls /content/processed/test/masks/   | wc -l

Mounted at /content/drive
=== Dataset Verification ===
Train images:
672
Train masks: 
672
Val images:  
149
Val masks:   
149
Test images: 
146
Test masks:  
146


In [2]:
# ============================================================
# CELL 4 - Clone Repository and Download Pretrained Weights
# ============================================================
import os
import shutil

repo_path = '/content/repo'

# 1. Move out of the directory before deleting it
%cd /content

# 2. Clean up existing repo safely
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
    print(f"Cleaned up existing repository at {repo_path}")

# 3. Clone the project repository
!git clone https://github.com/ilMassy/advertising-panel-segmentation {repo_path}

# 4. Change directory into the new repo
%cd {repo_path}

# 5. Create necessary directories
os.makedirs(os.path.join(repo_path, 'results'), exist_ok=True)
os.makedirs(os.path.join(repo_path, 'models'), exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)  # fuori dal repo

# 6. Download MIT-B1 pretrained weights
!wget -q --show-progress \
    https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b1_20220624-02e5a6a1.pth \
    -O /content/checkpoints/mit_b1_imagenet.pth

print("\nSetup completed successfully!")
!ls -lh /content/checkpoints/

/content
Cloning into '/content/repo'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 305 (delta 66), reused 72 (delta 30), pack-reused 190 (from 1)
Receiving objects: 100% (305/305), 348.35 KiB | 2.62 MiB/s, done.
Resolving deltas: 100% (164/164), done.
/content/repo
/content/checkpoint 100%[===================>]  50.22M  25.6MB/s    in 2.0s    

Setup completed successfully!
total 51M
-rw-r--r-- 1 root root 51M Jun 24  2022 mit_b1_imagenet.pth


In [3]:
# ============================================================
# CELL 5 - Verify Repository Structure
# ============================================================

import os

# Verify all files were cloned correctly
!echo "=== Config files ===" && ls /content/repo/configs/
!echo "=== Source scripts ===" && ls /content/repo/src/
!echo "=== Results directory ===" && ls /content/repo/results/
!echo "=== Models directory ===" && ls /content/repo/models/

# Verify dataset
!echo "=== Dataset ==="
!ls /content/processed/

print("Repository structure verified!")

=== Config files ===
segformer_b0_baseline.py   segformer_b1_optimized.py
segformer_b1_augmented.py  segformer_b1_standard.py
=== Source scripts ===
check_masks.py	CVAT_preparation.py  extract_frames.py	reorder_by_prefix.py
=== Results directory ===
exp0_segformer_b0_baseline  exp2_segformer_b1_augmented
exp1_segformer_b1_standard  exp3_segformer_b1_optimized
=== Models directory ===
README.md
=== Dataset ===
test  train  val
Repository structure verified!


In [4]:
# ============================================================
# CELL 6 - Create Optimized Config
# SegFormer-B1 with refined strategy:
# - LR 3e-5 for stable convergence
# - Refined Albumentations (Sharper edges, CoarseDropout)
# - Pure CrossEntropy Loss (Pixel-perfect precision)
# ============================================================


import os
os.makedirs('/content/repo/configs', exist_ok=True)

segformer_b1_optimized2_config = """
_base_ = [
    'mmseg::_base_/models/segformer_mit-b0.py',
    'mmseg::_base_/default_runtime.py',
]

# Dataset settings
dataset_type = 'BaseSegDataset'
data_root    = '/content/processed/'
crop_size    = (640, 640)

# Training pipeline (same augmentations as Phase 4)
train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='RandomResize', scale=(1920, 1080), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=crop_size, cat_max_ratio=0.75),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PhotoMetricDistortion'),
    dict(
        type='Albu',
        transforms=[
            dict(type='MotionBlur', blur_limit=5, p=0.3),
            dict(type='GaussNoise', var_limit=(5, 30), p=0.2),
            dict(type='RandomBrightnessContrast',
                 brightness_limit=0.3,
                 contrast_limit=0.3, p=0.4),
            dict(type='HueSaturationValue',
                 hue_shift_limit=20,
                 sat_shift_limit=30,
                 val_shift_limit=20, p=0.3),
            dict(type='CoarseDropout',
                 max_holes=4,
                 max_height=60,
                 max_width=60,
                 min_holes=1,
                 fill_value=0, p=0.25),
        ],
        keymap={'img': 'image', 'gt_semantic_seg': 'mask'},
        update_pad_shape=False,
    ),
    dict(type='PackSegInputs'),
]

# Validation/Test pipeline (no augmentation)
val_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=(1920, 1080), keep_ratio=True),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]

# Dataset meta info (2 classes: background and board)
meta = dict(
    classes=('background', 'board'),
    palette=[[0, 0, 0], [255, 0, 0]]
)

# Dataloaders
train_dataloader = dict(
    batch_size=4, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='train/images', seg_map_path='train/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=train_pipeline,
    )
)

val_dataloader = dict(
    batch_size=1, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='val/images', seg_map_path='val/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=val_pipeline,
    )
)

# Test set separated from validation
test_dataloader = dict(
    batch_size=1, num_workers=2,
    dataset=dict(
        type=dataset_type, data_root=data_root,
        data_prefix=dict(img_path='test/images', seg_map_path='test/masks'),
        img_suffix='.jpeg', seg_map_suffix='.png',
        metainfo=meta, pipeline=val_pipeline,
    )
)

# Evaluation metrics
val_evaluator  = dict(type='IoUMetric', iou_metrics=['mIoU', 'mDice'])
test_evaluator = dict(type='IoUMetric', iou_metrics=['mIoU', 'mDice', 'mFscore'])

# Model: SegFormer-B1
# Only change: LR 3e-5 instead of 6e-5
model = dict(
    data_preprocessor=dict(
        type='SegDataPreProcessor',
        mean=[123.675, 116.28, 103.53],
        std=[58.395, 57.12, 57.375],
        bgr_to_rgb=True,
        pad_val=0,
        seg_pad_val=255,
        size=crop_size,
        size_divisor=None,
    ),
    backbone=dict(
        embed_dims=64,
        num_heads=[1, 2, 5, 8],
        num_layers=[2, 2, 2, 2],
        drop_path_rate=0.1,
        init_cfg=dict(
            type='Pretrained',
            checkpoint='/content/checkpoints/mit_b1_imagenet.pth'
        )
    ),
    decode_head=dict(
        in_channels=[64, 128, 320, 512],
        num_classes=2,
        channels=256,        # standard decoder capacity
        dropout_ratio=0.1,
        loss_decode=dict(
            type='CrossEntropyLoss',
            loss_weight=1.0,
            use_sigmoid=False
        ),
    ),
    test_cfg=dict(mode='whole')
)

# AdamW optimizer - lower LR for stable convergence (only architectural change)
optimizer = dict(type='AdamW', lr=3e-5, betas=(0.9, 0.999), weight_decay=0.01)
optim_wrapper = dict(
    type='OptimWrapper',
    optimizer=optimizer,
    paramwise_cfg=dict(
        custom_keys={
            'pos_block': dict(decay_mult=0.),
            'norm':      dict(decay_mult=0.),
            'head':      dict(lr_mult=10.)
        }
    )
)

# LR scheduler with warmup
param_scheduler = [
    dict(type='LinearLR', start_factor=1e-6, by_epoch=False, begin=0,    end=1500),
    dict(type='PolyLR',   eta_min=0.0,       power=1.0,      begin=1500, end=20000, by_epoch=False),
]

# Save best checkpoint based on mIoU
default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', by_epoch=False, interval=2000,
                    max_keep_ckpts=3, save_best='mIoU', rule='greater'),
    logger=dict(type='LoggerHook', interval=50),
)

# Early stopping: stop if val mIoU does not improve for 5 consecutive validations
custom_hooks = [
    dict(
        type='EarlyStoppingHook',
        monitor='mIoU',
        rule='greater',
        min_delta=0.001,
        patience=5,
    )
]

train_cfg = dict(type='IterBasedTrainLoop', max_iters=20000, val_interval=2000)
val_cfg   = dict(type='ValLoop')
test_cfg  = dict(type='TestLoop')
"""

# Write config to disk
with open('/content/repo/configs/segformer_b1_optimized2.py', 'w') as f:
    f.write(segformer_b1_optimized2_config)

print(f"Config successfully created!")
print("\nPhase 5 Summary (Strategic Optimizations):")
print("-" * 50)
print("1. LR: 6e-5 -> 3e-5          (Stable convergence, lower overfitting)")
print("2. Decoder channels: 512->256 (Standard capacity, reduced overfitting)")
print("3. Augmentation: Lightened   (Reduced Blur/Noise, Modified CoarseDropout)")
print("4. Loss: Pure CrossEntropy   (Focusing on pixel-wise edge precision)")
print("-" * 50)
!ls -lh /content/repo/configs/

Config successfully created!

Phase 5 Summary (Strategic Optimizations):
--------------------------------------------------
1. LR: 6e-5 -> 3e-5          (Stable convergence, lower overfitting)
2. Decoder channels: 512->256 (Standard capacity, reduced overfitting)
3. Augmentation: Lightened   (Reduced Blur/Noise, Modified CoarseDropout)
4. Loss: Pure CrossEntropy   (Focusing on pixel-wise edge precision)
--------------------------------------------------
total 32K
-rw-r--r-- 1 root root 3.7K Mar 28 12:07 segformer_b0_baseline.py
-rw-r--r-- 1 root root 5.0K Mar 28 12:07 segformer_b1_augmented.py
-rw-r--r-- 1 root root 5.1K Mar 28 12:08 segformer_b1_optimized2.py
-rw-r--r-- 1 root root 5.1K Mar 28 12:07 segformer_b1_optimized.py
-rw-r--r-- 1 root root 3.9K Mar 28 12:07 segformer_b1_standard.py


In [5]:
# ============================================================
# CELL 7 - Push optimized Config to GitHub
# ============================================================

import os
from getpass import getpass

# Git configuration
!cd /content/repo && git config user.email "massimilianogiangreco7@gmail.com"
!cd /content/repo && git config user.name "ilMassy"

# Ask for token securely (input is hidden)
GITHUB_TOKEN = getpass("Enter your GitHub Personal Access Token: ")

# Set remote URL with authentication token
!cd /content/repo && git remote set-url origin \
    https://{GITHUB_TOKEN}@github.com/ilMassy/advertising-panel-segmentation.git

# Pull remote changes first
!cd /content/repo && git pull origin main --rebase

# Add and commit config file
!cd /content/repo && git add configs/
!cd /content/repo && git commit -m "Add SegFormer-B1 optimized config (Phase 5)" \
    || echo "Nothing to commit, already done"

# Push to GitHub
!cd /content/repo && git push origin main

print("Config file pushed to GitHub!")

Enter your GitHub Personal Access Token: ··········
From https://github.com/ilMassy/advertising-panel-segmentation
 * branch            main       -> FETCH_HEAD
Already up to date.
[main c754af9] Add SegFormer-B1 optimized config (Phase 5)
 1 file changed, 172 insertions(+)
 create mode 100644 configs/segformer_b1_optimized2.py
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 2.20 KiB | 2.20 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ilMassy/advertising-panel-segmentation.git
   7850101..c754af9  main -> main
Config file pushed to GitHub!


In [ ]:
# ============================================================
# CELL 8 - Training: SegFormer-B1 Optimized (Phase 5)
# Checkpoints saved directly to Google Drive during training
# ============================================================

import os
import mmseg

# Correct path for mmseg tools on Colab
mmseg_tools = os.path.join(
    os.path.dirname(mmseg.__file__),
    '.mim', 'tools', 'train.py'
)
print(f"MMSeg train tool: {mmseg_tools}")

# Verify tool exists
assert os.path.exists(mmseg_tools), f"train.py not found at {mmseg_tools}"
print("Train tool found!")

# Create output directory on Drive
!mkdir -p /content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2

# Launch B1 optimized training
# work-dir points directly to Drive so checkpoints are saved during training
!python {mmseg_tools} \
    /content/repo/configs/segformer_b1_optimized2.py \
    --work-dir /content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2 \
    2>&1 | tee /content/training_b1_optimized2_log.txt

MMSeg train tool: /usr/local/lib/python3.12/dist-packages/mmseg/.mim/tools/train.py
Train tool found!
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
03/27 13:08:18 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 546033338
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Inte

In [6]:
# ============================================================
# CELL 9 - Verify Optimized Results on Google Drive
# ============================================================

drive_results = "/content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2"

print("Checkpoints saved on Drive:")
!ls -lh {drive_results}/*.pth 2>/dev/null || echo "  No checkpoints yet"

print("\nLast checkpoint:")
!cat {drive_results}/last_checkpoint 2>/dev/null || echo "  Not found"

Checkpoints saved on Drive:
-rw------- 1 root root  54M Mar 27 15:22 /content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2/best_mIoU_iter_12000.pth
-rw------- 1 root root 159M Mar 27 16:05 /content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2/iter_16000.pth
-rw------- 1 root root 160M Mar 27 16:27 /content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2/iter_18000.pth
-rw------- 1 root root 160M Mar 27 16:50 /content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2/iter_20000.pth

Last checkpoint:
/content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2/iter_20000.pth

In [ ]:
# ============================================================
# CELL 10 - Evaluate B1 Optimized with Full Metrics
# mIoU, mDice, Precision, Recall on TEST SET
# Uses best checkpoint saved on Google Drive
# ============================================================

import mmseg, os, glob

mmseg_test = os.path.join(
    os.path.dirname(mmseg.__file__),
    '.mim', 'tools', 'test.py'
)

# Automatically find best checkpoint on Drive
checkpoints = glob.glob(
    '/content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2/best_mIoU_iter_*.pth'
)

if not checkpoints:
    print("No checkpoint found! Check Drive path.")
else:
    best_checkpoint = max(
        checkpoints,
        key=lambda x: int(x.split('iter_')[1].split('.')[0])
    )
    print(f"Using checkpoint: {best_checkpoint}")

    # test_dataloader  → test/images + test/masks (146 immagini)
    # test_evaluator   → mIoU, mDice, mFscore (Precision + Recall)
    !python {mmseg_test} \
        /content/repo/configs/segformer_b1_optimized2.py \
        {best_checkpoint}

Using checkpoint: /content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2/best_mIoU_iter_12000.pth
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
03/27 16:52:02 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1334192523
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.4.1+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3

In [11]:
# ============================================================
# CELL 11 - Save Logs and Push to GitHub
# Copies B1 optimized training logs from Drive to GitHub repo
# ============================================================

import os, glob, shutil

# ── Create directories ────────────────────────────────────────────────────────
os.makedirs('/content/repo/results/exp3_segformer_b1_optimized2', exist_ok=True)
os.makedirs('/content/repo/models', exist_ok=True)

# ── Copy training logs from Drive to results/ ─────────────────────────────────
src = '/content/drive/MyDrive/advertising_panel_segmentation/results/exp3_segformer_b1_optimized2'
dst = '/content/repo/results/exp3_segformer_b1_optimized2'

log_files = glob.glob(f'{src}/**/*.log', recursive=True)
if log_files:
    for log_file in log_files:
        shutil.copy(log_file, dst)
        print(f"Copied: {os.path.basename(log_file)} → results/exp3_segformer_b1_optimized2/")
else:
    print(f"No log files found in {src}")

# ── Update models/README.md ───────────────────────────────────────────────────
readme_path = '/content/repo/models/README.md'
existing = ""
if os.path.exists(readme_path):
    with open(readme_path, 'r') as f:
        existing = f.read()

b1_opt2_entry = """
## Exp3 - SegFormer-B1 Optimized2
- Best checkpoint: best_mIoU_iter_12000.pth
- mIoU: 84.92%
- board IoU: 72.26%
- Dice: 83.90%
- Precision: 80.40%
- Recall: 87.71%
- Stored on: Google Drive
"""

if 'Exp3 - SegFormer-B1 Optimized2' not in existing:
    with open(readme_path, 'a') as f:
        f.write(b1_opt2_entry)
    print("Updated models/README.md with Exp3 entry")
else:
    print("Exp3 entry already present in models/README.md")

# ── Verify no .pth files exist ───────────────────────────────────────────────
pth_files = glob.glob('/content/repo/results/**/*.pth', recursive=True)
if pth_files:
    print("WARNING: .pth files found! Removing...")
    for f in pth_files:
        os.remove(f)
        print(f"Removed: {f}")
else:
    print("No .pth files found - safe to push!")

# ── Push to GitHub ────────────────────────────────────────────────────────────
from getpass import getpass
GITHUB_TOKEN = getpass("Enter GitHub token: ")

!cd /content/repo && git config user.email "massimilianogiangreco7@gmail.com"
!cd /content/repo && git config user.name "ilMassy"
!cd /content/repo && git remote set-url origin \
    https://{GITHUB_TOKEN}@github.com/ilMassy/advertising-panel-segmentation.git
!cd /content/repo && git pull origin main --rebase

!cd /content/repo && git add results/exp3_segformer_b1_optimized2/*.log
!cd /content/repo && git add models/README.md

print("\n=== Files to be committed ===")
!cd /content/repo && git diff --cached --name-only

!cd /content/repo && git commit -m "Add B1 optimized training logs and model summary (Phase 5)" \
    || echo "Nothing to commit"
!cd /content/repo && git push origin main
print("Pushed to GitHub!")

Copied: 20260327_130817.log → results/exp3_segformer_b1_optimized2/
Exp3 entry already present in models/README.md
No .pth files found - safe to push!
Enter GitHub token: ··········
From https://github.com/ilMassy/advertising-panel-segmentation
 * branch            main       -> FETCH_HEAD
Already up to date.

=== Files to be committed ===
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Nothing to commit
Everything up-to-date
Pushed to GitHub!
